# Notebook to compute exact hyperbolicity of graphs used in the paper using SageMath.

In [1]:
from sage.all import *
from sage.graphs.hyperbolicity import hyperbolicity_distribution, hyperbolicity
# simple test
G = graphs.GridGraph([Integer(3),Integer(3)])
hyperbolicity_distribution(G,algorithm='exact')

{0: 11/18, 1: 8/21, 2: 1/126}

In [ ]:
import os
import pickle as pkl
import numpy as np
import networkx as nx
import scipy.sparse as sp

In [3]:
def load_planetoid(path, dataset="cora"):
    prefix=f"ind.{dataset.lower()}"
    def parse_index_file(filename):
        with open(filename) as f:
            return [int(line.strip()) for line in f]

    def load_pickle(filename):
        with open(filename, 'rb') as f:
            return pkl.load(f, encoding='latin1')

    # Load the data
    x = load_pickle(os.path.join(path, f"{prefix}.x")).toarray()
    tx = load_pickle(os.path.join(path, f"{prefix}.tx")).toarray()
    allx = load_pickle(os.path.join(path, f"{prefix}.allx")).toarray()
    y = load_pickle(os.path.join(path, f"{prefix}.y"))
    ty = load_pickle(os.path.join(path, f"{prefix}.ty"))
    ally = load_pickle(os.path.join(path, f"{prefix}.ally"))
    graph = load_pickle(os.path.join(path, f"{prefix}.graph"))
    test_idx = parse_index_file(os.path.join(path, f"{prefix}.test.index"))

    test_idx_sorted = np.sort(test_idx)
    num_nodes = max(graph.keys()) + 1
    feature_dim = allx.shape[1]

    # Create full feature and label matrices
    features = np.zeros((num_nodes, feature_dim), dtype=np.float32)
    labels = np.zeros((num_nodes, y.shape[1]), dtype=np.int64)

    # Fill training and validation data
    features[:allx.shape[0], :] = allx
    labels[:ally.shape[0], :] = ally

    # Fill test data
    for i, idx in enumerate(test_idx):
        features[idx] = tx[i]
        labels[idx] = ty[i]

    # Build graph
    G = nx.from_dict_of_lists(graph)
    G.remove_edges_from(nx.selfloop_edges(G))

    return G, features, labels.argmax(axis=1)


In [6]:
def get_largest_connected_component(G_nx):
    # Convert to undirected (if not already)
    G_undirected = G_nx.to_undirected()

    # Get largest connected component
    largest_cc_nodes = max(nx.connected_components(G_undirected), key=len)
    G_lcc = G_undirected.subgraph(largest_cc_nodes).copy()
    return G_lcc

def get_kth_largest_connected_component(G_nx, k=1):
    # Convert to undirected if not already
    G_undirected = G_nx.to_undirected()

    # Get all connected components, sorted by size (descending)
    components = sorted(nx.connected_components(G_undirected), key=len, reverse=True)

    if k <= 0 or k > len(components):
        raise ValueError(f"Invalid k={k}: graph has only {len(components)} connected components")

    # Extract the k-th largest component (1-based index)
    kth_cc_nodes = components[k - 1]
    G_kcc = G_undirected.subgraph(kth_cc_nodes).copy()
    return G_kcc


## Example: How to compute hyperbolicity if there is more than one connected component for Cora dataset

In [35]:
G, X, y = load_planetoid("data/cora/cora/raw")

print("Graph info:", G)
print("Features shape:", X.shape)
print("Labels shape:", y.shape)

for k in range(1, 11):
    try:
        G_klcc = get_kth_largest_connected_component(G, k)
        print(G_klcc)
        G_klcc_sage = Graph(G_klcc)
        delta, certificate, delta_UB = hyperbolicity(G_klcc_sage, algorithm='BCCM')
        print(f"Hyperbolicity in {k}th largest component:", delta)
    except ValueError as e:
        print(e)



Graph with 26 nodes and 43 edges
Hyperbolicity in 2th largest component: (1, [145, 1327, 144, 2160], 1)
Time taken: 0.00 seconds
Graph with 9 nodes and 23 edges
Hyperbolicity in 3th largest component: (1, [583, 2222, 75, 284], 1)
Time taken: 0.00 seconds
Graph with 8 nodes and 15 edges
Hyperbolicity in 4th largest component: (1/2, [26, 2604, 2454, 2455], 1/2)
Time taken: 0.00 seconds
Graph with 6 nodes and 8 edges
Hyperbolicity in 5th largest component: (1/2, [2438, 167, 2437, 168], 1/2)
Time taken: 0.00 seconds
Graph with 5 nodes and 7 edges
Hyperbolicity in 6th largest component: (1/2, [2661, 2662, 1318, 1001], 1/2)
Time taken: 0.00 seconds
Graph with 5 nodes and 5 edges
Hyperbolicity in 7th largest component: (0, [], 0)
Time taken: 0.00 seconds
Graph with 5 nodes and 7 edges
Hyperbolicity in 8th largest component: (1/2, [2688, 2686, 1657, 2687], 1/2)
Time taken: 0.00 seconds
Graph with 4 nodes and 4 edges
Hyperbolicity in 9th largest component: (0, [], 0)
Time taken: 0.00 seconds
Gr

# Compute hyperbolicity for Planetoid datasets

In [37]:
datasets = ['Cora', 'Citeseer', 'PubMed']
dataset_folder = ["data/Cora/Cora/raw",
                "data/Citeseer/Citeseer/raw",
                "data/PubMed/PubMed/raw"]
for i, dataset in enumerate(datasets):
    G, X, y = load_planetoid(dataset_folder[i], dataset=dataset)
    print("Graph info:", G)
    G_lcc = get_largest_connected_component(G)
    print(f"Dataset: {dataset}, Largest Connected Component: {G_lcc}")
    G_lcc_sage = Graph(G_lcc)
    delta, certificate, delta_UB = hyperbolicity(G_lcc_sage, algorithm='BCCM')
    print(f"Hyperbolicity for {dataset}:", delta)

Graph info: Graph with 2708 nodes and 5278 edges
Dataset: Cora, Largest Connected Component: Graph with 2485 nodes and 5069 edges
Hyperbolicity for Cora: 4 4
Graph info: Graph with 3327 nodes and 4552 edges
Dataset: Citeseer, Largest Connected Component: Graph with 2120 nodes and 3679 edges
Hyperbolicity for Citeseer: 13/2 13/2
Graph info: Graph with 19717 nodes and 44324 edges
Dataset: PubMed, Largest Connected Component: Graph with 19717 nodes and 44324 edges
Hyperbolicity for PubMed: 9/2 9/2


In [39]:
def load_data_airport(data_path, augment_features=False):
    path = os.path.join(data_path, 'airport/airport.p')
    with open(path, 'rb') as f:
        graph = pkl.load(f)

    # Create consistent node ordering
    nodes = list(graph.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    num_nodes = len(nodes)

    # Compute adjacency matrix
    row, col = [], []
    for u, v in graph.edges():
        row.append(node_to_idx[u])
        col.append(node_to_idx[v])
    adj = sp.coo_matrix((np.ones(len(row)), (row, col)), shape=(num_nodes, num_nodes)).tocsr()

    # Extract original features and labels
    features = np.array([graph.nodes[u]['feat'] for u in nodes])
    label_idx = 4
    labels = features[:, label_idx]
    base_features = features[:, :label_idx]

    # Augment features if requested
    if augment_features:
        deg = np.squeeze(np.sum(adj, axis=0).astype(int))
        deg = np.clip(deg, 0, 5)
        deg_onehot = np.eye(6)[deg].squeeze()
        const_f = np.ones((base_features.shape[0], 1))
        augmented_features = np.concatenate((base_features, deg_onehot, const_f), axis=1)
    else:
        augmented_features = base_features

    # Digitize labels
    labels = np.digitize(labels, bins=[7.0 / 7, 8.0 / 7, 9.0 / 7])

    # Add features and labels back to graph nodes
    for i, node in enumerate(nodes):
        graph.nodes[node]['features'] = augmented_features[i]
        graph.nodes[node]['label'] = labels[i]

    graph.remove_edges_from(nx.selfloop_edges(graph))
    return graph

# Compute hyperbolicity for Airport dataset

In [41]:
G = load_data_airport("../data", augment_features=True)
print("Graph info:", G)
G_lcc = get_largest_connected_component(G)
print(f"Dataset: Airport, Largest Connected Component: {G_lcc}")
G_lcc_sage = Graph(G_lcc)
delta, certificate, delta_UB = hyperbolicity(G_lcc_sage, algorithm='BCCM')
print(f"Hyperbolicity for Airport:", delta)

Graph info: Graph with 3188 nodes and 18630 edges
Dataset: Airport, Largest Connected Component: Graph with 3158 nodes and 18605 edges
Hyperbolicity for Airport: 2


In [19]:
def load_disease_data(dataset_str, data_dir, use_feats=True):
    object_to_idx = {}
    idx_counter = 0
    edges = []

    data_path = os.path.join(data_dir, dataset_str)

    # Parse edge list and map node names to indices
    with open(os.path.join(data_path, f"{dataset_str}.edges.csv"), 'r') as f:
        all_edges = f.readlines()

    for line in all_edges:
        n1, n2 = line.rstrip().split(',')

        if n1 not in object_to_idx:
            object_to_idx[n1] = idx_counter
            idx_counter += 1
        if n2 not in object_to_idx:
            object_to_idx[n2] = idx_counter
            idx_counter += 1

        i = object_to_idx[n1]
        j = object_to_idx[n2]
        edges.append((i, j))

    num_nodes = len(object_to_idx)

    # Load features
    if use_feats:
        features = sp.load_npz(os.path.join(data_path, f"{dataset_str}.feats.npz")).toarray()
    else:
        features = np.eye(num_nodes)

    # Load labels
    labels = np.load(os.path.join(data_path, f"{dataset_str}.labels.npy"))

    # Build graph
    G = nx.Graph()
    G.add_edges_from(edges)

    for i in range(num_nodes):
        G.add_node(i, features=features[i], label=int(labels[i]))

    return G

# Compute hyperbolicity for Disease and Synthetic Tree datasets

In [20]:
datasets = ['disease_nc', 'disease_lp', 'disease_m_lp', 'tree1111_g00_lp']
for i, dataset in enumerate(datasets):
    G = load_disease_data(dataset, "../data", use_feats=True)
    print("Graph info:", G)
    G_lcc = get_largest_connected_component(G)
    print(f"Dataset: {dataset}, Largest Connected Component: {G_lcc}")
    G_lcc_sage = Graph(G_lcc)
    delta, certificate, delta_UB = hyperbolicity(G_lcc_sage, algorithm='BCCM')
    print(f"Hyperbolicity for {dataset}:", delta)


Graph info: Graph with 1044 nodes and 1043 edges
Dataset: disease_nc, Largest Connected Component: Graph with 1044 nodes and 1043 edges
Hyperbolicity for disease_nc: 0
Graph info: Graph with 2665 nodes and 2664 edges
Dataset: disease_lp, Largest Connected Component: Graph with 2665 nodes and 2664 edges
Hyperbolicity for disease_lp: 0
Graph info: Graph with 2529 nodes and 2528 edges
Dataset: disease_m_lp, Largest Connected Component: Graph with 2529 nodes and 2528 edges
Hyperbolicity for disease_m_lp: 0
Graph info: Graph with 1111 nodes and 1110 edges
Dataset: tree1111_g00_lp, Largest Connected Component: Graph with 1111 nodes and 1110 edges
Hyperbolicity for tree1111_g00_lp: 0
Graph info: Graph with 1111 nodes and 1110 edges
Dataset: tree1111_g10_lp, Largest Connected Component: Graph with 1111 nodes and 1110 edges
Hyperbolicity for tree1111_g10_lp: 0
Graph info: Graph with 3000 nodes and 14975 edges
Dataset: sf_3000_lp, Largest Connected Component: Graph with 3000 nodes and 14975 edge